In [28]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Список для хранения данных
articles_data = []

# Перебор страниц
for page in range(1, 11):  # Перебор первых десяти страниц
    # Формирование URL для каждой страницы
    url = f"https://lifehacker.ru/topics/technology/?page={page}" if page != 1 else "https://lifehacker.ru/topics/technology/"
    print(f'Обрабатывается страница: {url}')

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except Exception as e:
        print(f"Не удалось получить доступ к странице {page}: {e}")
        continue

    soup = BeautifulSoup(response.text, 'html.parser')

    # Поиск ссылок на статьи с классом lh-small-article-card__link
    article_links = soup.find_all('a', class_='lh-small-article-card__link')

    # Извлечение ссылок на статьи
    article_urls = [link.get('href') for link in article_links if link.get('href')]

    # Перебор ссылок на статьи
    for article_url in article_urls:
        # Проверка ссылки
        if article_url.startswith('http'):
            full_article_url = article_url
        else:
            full_article_url = f"https://lifehacker.ru{article_url}"

        try:
            article_response = requests.get(full_article_url, timeout=10)
            article_response.raise_for_status()
            article_soup = BeautifulSoup(article_response.text, 'html.parser')

            # Поиск заголовка статьи в теге <title>
            title_tag = article_soup.find('title')
            title = title_tag.get_text().strip() if title_tag else 'No title found'

            # Поиск основного текста статьи в тегах <p> с атрибутами data-v-36e8b263 и data-v-9cdbc6f5
            content_paragraphs = article_soup.find_all('p', attrs={"data-v-36e8b263": True}) + article_soup.find_all('p', attrs={"data-v-9cdbc6f5": True})
            content = ' '.join([p.get_text().strip() for p in content_paragraphs if p.get_text().strip()])

            if not content:
                content = 'No content found'

            # Добавление данных в список
            articles_data.append({'title': title, 'content': content})
            print(f"Добавлена статья: {title}")
        except Exception as e:
            print(f"Ошибка при обработке {full_article_url}: {e}")

        time.sleep(1)  # Пауза между запросами

# Создание датафрейма
df = pd.DataFrame(articles_data)

# Сохранение датафрейма в файл csv
df.to_csv('articles.csv', index=False, encoding='utf-8')

print("Данные успешно сохранены в файл articles.csv")


Обрабатывается страница: https://lifehacker.ru/topics/technology/
Добавлена статья: 5 новых игр для Android и iOS: лучшее за май — Лайфхакер
Добавлена статья: 10 лучших водонагревателей для дачи и квартиры — Лайфхакер
Добавлена статья: В Греции пройдёт первая в истории международная Олимпиада среди роботов — Лайфхакер
Добавлена статья: 9 новых приложений для iOS: лучшее за май — Лайфхакер
Добавлена статья: Объясняем за минуту: может ли чат-бот иметь сознание и сводить вас с ума — Лайфхакер
Добавлена статья: Обзор смарт-часов HUAWEI WATCH FIT 4 Pro и сравнение с WATCH FIT 4: эволюция и апгрейд — Лайфхакер
Добавлена статья: Появились первые подробности о Xiaomi 16 — со Snapdragon 8 Elite 2 и 100-ваттной зарядкой — Лайфхакер
Добавлена статья: Лучшие смартфоны мая — Лайфхакер
Добавлена статья: Какие гаджеты и аксессуары мы берём в отпуск: советует редакция Лайфхакера — Лайфхакер
Добавлена статья: Windows 11 сможет сама обновлять все приложения на ПК — даже сторонние — Лайфхакер
Добавлена с